In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


json_path = Path(
    "../../../1_cellvit/output/cellvit_out/SPA51/cell_detection.json"
)

if not json_path.exists():
    raise FileNotFoundError(f"File not found: {json_path}")

with json_path.open("r") as f:
    cellvit_data = json.load(f)

print(cellvit_data.keys())

cells = cellvit_data["cells"]

print("Number of cells:", len(cells))
print("First cell:", cells[0])


type_map = {
    int(type_id): cell_type
    for type_id, cell_type in cellvit_data["type_map"].items()
}

print(type_map)

obs_rows = []

for cell_number, cell in enumerate(cells):

    # Create a stable, unique cell identifier
    cell_id = f"cell_{cell_number:06d}"

    # CellViT centroid is stored as [x, y]
    centroid_x, centroid_y = cell["centroid"]

    # CellViT bbox is stored as:
    # [[y_min, x_min], [y_max, x_max]]
    bbox_ymin, bbox_xmin = cell["bbox"][0]
    bbox_ymax, bbox_xmax = cell["bbox"][1]

    # Numeric CellViT class
    cell_type_id = int(cell["type"])

    # Convert numeric class to readable phenotype
    cell_type = type_map.get(cell_type_id, "Unknown")

    obs_rows.append({
        "cell_id": cell_id,
        "x": float(centroid_x),
        "y": float(centroid_y),
        "cell_type_id": cell_type_id,
        "cell_type": cell_type,
        "bbox_xmin": float(bbox_xmin),
        "bbox_ymin": float(bbox_ymin),
        "bbox_xmax": float(bbox_xmax),
        "bbox_ymax": float(bbox_ymax)
    })

obs_df = pd.DataFrame(obs_rows)
print(obs_df.head())

obs_df = obs_df.set_index("cell_id")

print("obs shape:", obs_df.shape)
print("index name:", obs_df.index.name)
print("unique cell IDs:", obs_df.index.is_unique)
print("missing coordinates:", obs_df[["x", "y"]].isna().sum().to_dict())

centroid_inside_bbox = (
    (obs_df["x"] >= obs_df["bbox_xmin"]) &
    (obs_df["x"] <= obs_df["bbox_xmax"]) &
    (obs_df["y"] >= obs_df["bbox_ymin"]) &
    (obs_df["y"] <= obs_df["bbox_ymax"])
)

print(
    "Centroids inside bounding boxes:",
    f"{centroid_inside_bbox.mean():.2%}"
)

print(obs_df["cell_type"].value_counts())

In [ ]:
all_type_counts = (
    obs_df["cell_type_id"]
    .value_counts()
    .reindex(type_map.keys(), fill_value=0)
    .rename(index=type_map)
)

print(all_type_counts)

## Morphology features

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from shapely.geometry import Polygon, MultiPolygon, shape
from skimage.draw import polygon as draw_polygon
from skimage.measure import regionprops

In [ ]:
print(obs_df.shape)
print(obs_df.head())

In [ ]:
geojson_path = Path(
    "../../../1_cellvit/output/cellvit_out/SPA51/cells.geojson"
)

if not geojson_path.exists():
    raise FileNotFoundError(
        f"GeoJSON file not found: {geojson_path}"
    )

with geojson_path.open("r") as f:
    geojson = json.load(f)

print(type(geojson))



In [ ]:
feature = geojson[0]

print("Feature keys:")
print(feature.keys())

print("\nProperties:")
print(feature["properties"])

print("\nGeometry type:")
print(feature["geometry"]["type"])

coords = feature["geometry"]["coordinates"]

print("\nCoordinates type:")
print(type(coords))

print("\nNumber of coordinate elements:")
print(len(coords))

print("\nFirst few coordinates:")
print(coords[0][0][:5])

In [ ]:
for feature in geojson:

    cell_type = (
        feature["properties"]
        ["classification"]
        ["name"]
    )

    geometry_type = feature["geometry"]["type"]

    n_polygons = len(
        feature["geometry"]["coordinates"]
    )

    print(
        cell_type,
        geometry_type,
        n_polygons
    )

In [ ]:
coords = geojson[0]["geometry"]["coordinates"]

print("Number of polygons:", len(coords))

polygon = coords[0]
print("Polygon type:", type(polygon))
print("Number of rings:", len(polygon))

ring = polygon[0]
print("Ring type:", type(ring))
print("Number of contour points:", len(ring))

print("\nFirst 10 contour points:")
for point in ring[:10]:
    print(point)

In [ ]:
polygon = geojson[0]["geometry"]["coordinates"][0]
contour = polygon[0]

print("Number of contour points:", len(contour))
print("First five points:", contour[:5])

In [ ]:
def clean_contour(contour):
    """
    Convert a contour to a NumPy array and remove
    consecutive duplicate points.
    """

    points = np.asarray(
        contour,
        dtype=np.float64
    )

    if points.ndim != 2 or points.shape[1] != 2:
        return None

    if len(points) < 3:
        return None

    # Remove consecutive duplicate coordinates
    keep = np.ones(len(points), dtype=bool)

    keep[1:] = np.any(
        points[1:] != points[:-1],
        axis=1
    )

    points = points[keep]

    # At least three unique points are required
    if len(points) < 3:
        return None

    return points

In [ ]:
cleaned = clean_contour(contour)

print("Original points:", len(contour))
print("Cleaned points:", len(cleaned))
print(cleaned[:5])

In [ ]:
def contour_to_local_mask(points, padding=2):
    """
    Convert one cell contour into a small local binary mask.
    """

    x = points[:, 0]
    y = points[:, 1]

    x_min = int(np.floor(x.min())) - padding
    y_min = int(np.floor(y.min())) - padding

    x_max = int(np.ceil(x.max())) + padding
    y_max = int(np.ceil(y.max())) + padding

    width = x_max - x_min + 1
    height = y_max - y_min + 1

    if width <= 0 or height <= 0:
        return None

    mask = np.zeros(
        (height, width),
        dtype=np.uint8
    )

    local_x = x - x_min
    local_y = y - y_min

    rr, cc = draw_polygon(
        local_y,
        local_x,
        shape=mask.shape
    )

    mask[rr, cc] = 1

    return mask

In [ ]:
def calculate_morphology(contour):
    """
    Calculate morphology features for one CellViT contour.

    Returns
    -------
    dict or None

    Features returned
    -----------------
    area_px2
    perimeter_px
    circularity
    solidity
    eccentricity
    """

    points = clean_contour(contour)

    if points is None:
        return None

    # Construct a Shapely polygon from the contour
    poly = Polygon(points)

    # Attempt to repair invalid geometry
    if not poly.is_valid:
        poly = poly.buffer(0)

    if poly.is_empty:
        return None

    # Geometry repair may produce multiple polygons.
    # Retain the largest connected component.
    if isinstance(poly, MultiPolygon):
        poly = max(
            poly.geoms,
            key=lambda polygon: polygon.area
        )

    if not isinstance(poly, Polygon):
        return None

    # Polygon-based features
    area = float(poly.area)
    perimeter = float(poly.length)

    if area <= 0 or perimeter <= 0:
        return None

    # Circularity:
    # 1 indicates a theoretically perfect circle.
    circularity = (
        4.0 * np.pi * area /
        (perimeter ** 2)
    )

    # Solidity:
    # polygon area divided by convex-hull area
    convex_hull_area = float(poly.convex_hull.area)

    solidity = (
        area / convex_hull_area
        if convex_hull_area > 0
        else np.nan
    )

    # Eccentricity must be computed from a binary mask
    eccentricity = np.nan

    mask = contour_to_local_mask(points)

    if mask is not None and mask.any():
        regions = regionprops(mask)

        if len(regions) > 0:
            eccentricity = float(
                regions[0].eccentricity
            )

    return {
        "area_px2": area,
        "perimeter_px": perimeter,
        "circularity": circularity,
        "solidity": solidity,
        "eccentricity": eccentricity
    }

In [ ]:
test_features = calculate_morphology(contour)

print(test_features)

In [ ]:
morphology_rows = []
feature = geojson[0]

cell_type = (
    feature["properties"]
    ["classification"]
    ["name"]
)

cell_polygons = (
    feature["geometry"]
    ["coordinates"]
)

print(cell_type)
print(len(cell_polygons))

for polygon in cell_polygons:

    contour = polygon[0]

    features = calculate_morphology(contour)

    if features is None:
        continue

    features["cell_type"] = cell_type

    morphology_rows.append(features)

In [ ]:
morphology_df = pd.DataFrame(morphology_rows)
print(morphology_df.shape)

morphology_df.head()

In [ ]:
import matplotlib.pyplot as plt

features = [
    "area_px2",
    "perimeter_px",
    "circularity",
    "eccentricity",
    "solidity"
]

for feature in features:

    plt.figure(figsize=(5,3))

    plt.hist(
        morphology_df[feature],
        bins=50
    )

    plt.title(feature)

    plt.show()